# IBM SkillsBuild Cyber Security 2026 | Conducted by BharatCares
## Project: AI-Powered Network Intrusion Detection System (NIDS)
**Author:** Vaishnavi Gondal  
**Domain:** Cyber Security & Applied Machine Learning  
**Objective:** Build, train, and evaluate a multi-class and binary Network Intrusion Detection System capable of detecting malicious attacks (DoS, Probe, R2L, U2R) with high precision and minimal false alarm rates.

### Section 1: Environment Setup & Library Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-Learn Modules
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
import joblib

print("[✓] All core security & ML libraries successfully imported.")

### Section 2: Synthetic / Benchmark Network Traffic Data Generator
In production or offline benchmarking, the NSL-KDD or CICIDS2017 dataset is ingested. Here, we build a representative network traffic dataset with network protocol features, packet rates, error flags, and attack labels.

In [2]:
# Synthetic data generation simulating real-world network packet traces
np.random.seed(42)
n_samples = 5000

protocols = ['tcp', 'udp', 'icmp']
services = ['http', 'smtp', 'ftp', 'dns', 'ssh', 'telnet', 'private']
flags = ['SF', 'S0', 'REJ', 'RSTO', 'SH']
attack_types = ['Normal', 'DoS', 'Probe', 'R2L', 'U2R']

data = {
    'duration': np.random.exponential(scale=2.5, size=n_samples),
    'protocol_type': np.random.choice(protocols, size=n_samples, p=[0.75, 0.20, 0.05]),
    'service': np.random.choice(services, size=n_samples),
    'flag': np.random.choice(flags, size=n_samples, p=[0.70, 0.15, 0.08, 0.04, 0.03]),
    'src_bytes': np.random.gamma(shape=2, scale=500, size=n_samples).astype(int),
    'dst_bytes': np.random.gamma(shape=2, scale=1200, size=n_samples).astype(int),
    'wrong_fragment': np.random.choice([0, 1, 3], size=n_samples, p=[0.97, 0.02, 0.01]),
    'urgent': np.random.choice([0, 1], size=n_samples, p=[0.995, 0.005]),
    'count': np.random.poisson(lam=15, size=n_samples),
    'srv_count': np.random.poisson(lam=12, size=n_samples),
    'serror_rate': np.random.uniform(0.0, 1.0, size=n_samples),
    'rerror_rate': np.random.uniform(0.0, 0.5, size=n_samples),
    'same_srv_rate': np.random.uniform(0.1, 1.0, size=n_samples),
    'diff_srv_rate': np.random.uniform(0.0, 0.9, size=n_samples),
    'dst_host_count': np.random.randint(1, 255, size=n_samples),
    'dst_host_srv_count': np.random.randint(1, 255, size=n_samples),
    'attack_class': np.random.choice(attack_types, size=n_samples, p=[0.55, 0.25, 0.12, 0.05, 0.03])
}

df = pd.DataFrame(data)
# Inject realistic intrusion anomalies: High packet count and error rate for DoS
df.loc[df['attack_class'] == 'DoS', 'src_bytes'] = np.random.gamma(shape=5, scale=2500, size=(df['attack_class'] == 'DoS').sum())
df.loc[df['attack_class'] == 'DoS', 'count'] = np.random.randint(150, 512, size=(df['attack_class'] == 'DoS').sum())
df.loc[df['attack_class'] == 'DoS', 'serror_rate'] = np.random.uniform(0.8, 1.0, size=(df['attack_class'] == 'DoS').sum())

# Binary target column for high-speed triage
df['is_intrusion'] = (df['attack_class'] != 'Normal').astype(int)

print("[✓] Synthetic Network Security Dataset initialized successfully.")
df.head()

### Section 3: Exploratory Data Analysis & Threat Distribution

In [3]:
print("=== Dataset Summary ===")
print(df.info())
print("\n=== Attack Distribution ===")
print(df['attack_class'].value_counts())

# Statistical distribution
df.describe()

### Section 4: Data Preprocessing, Encoding & Feature Scaling

In [4]:
# Categorical Feature Encoding
categorical_cols = ['protocol_type', 'service', 'flag']
label_encoders = {}
df_encoded = df.copy()

for col in categorical_cols:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    label_encoders[col] = le

# Feature Selection
feature_cols = [c for c in df_encoded.columns if c not in ['attack_class', 'is_intrusion']]
X = df_encoded[feature_cols]
y_binary = df_encoded['is_intrusion']
y_multi = df_encoded['attack_class']

# Train-Test Split (80/20 Stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.20, random_state=42, stratify=y_binary
)

# Feature Scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training set shape: {X_train.shape}, Testing set shape: {X_test.shape}")

### Section 5: Machine Learning Model Training & Comparative Benchmarking

In [5]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
}

results = []

for name, clf in models.items():
    clf.fit(X_train_scaled, y_train)
    preds = clf.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    results.append({'Model': name, 'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1-Score': f1})

results_df = pd.DataFrame(results)
print("=== Model Performance Comparison ===")
print(results_df.to_string(index=False))

### Section 6: In-Depth Evaluation of the Champion Model (Random Forest)
Evaluating Confusion Matrix, ROC-AUC, and Feature Importance to detect cybersecurity attack signatures.

In [6]:
rf_model = models['Random Forest']
rf_preds = rf_model.predict(X_test_scaled)

print("=== Classification Report (Random Forest) ===")
print(classification_report(y_test, rf_preds, target_names=['Normal (Benign)', 'Attack (Intrusion)']))

# Feature Importance Analysis
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\n=== Top Cyber Attack Predictive Features ===")
print(importances.head(8))

### Section 7: Threat Simulation & Real-Time Incident Classification Pipeline

In [7]:
def inspect_packet_traffic(raw_packet_dict):
    """
    Simulates real-time deep packet metadata inspection and threat scoring.
    """
    temp_df = pd.DataFrame([raw_packet_dict])
    for c in categorical_cols:
        temp_df[c] = label_encoders[c].transform(temp_df[c])
    scaled_features = scaler.transform(temp_df[feature_cols])
    prediction = rf_model.predict(scaled_features)[0]
    prob = rf_model.predict_proba(scaled_features)[0][1]
    
    status = "CRITICAL ALERT: INTRUSION DETECTED!" if prediction == 1 else "NORMAL TRAFFIC: BENIGN"
    return {
        "Security_Verdict": status,
        "Intrusion_Probability": f"{prob * 100:.2f}%",
        "Recommended_Action": "Block IP via Firewall Rules & Alert SOC Analyst" if prediction == 1 else "Allow Traffic"
    }

# Sample Suspicious DoS Packet Simulation
suspicious_packet = {
    'duration': 0.05,
    'protocol_type': 'tcp',
    'service': 'private',
    'flag': 'S0',
    'src_bytes': 14500,
    'dst_bytes': 0,
    'wrong_fragment': 0,
    'urgent': 0,
    'count': 340,
    'srv_count': 320,
    'serror_rate': 0.98,
    'rerror_rate': 0.0,
    'same_srv_rate': 0.95,
    'diff_srv_rate': 0.02,
    'dst_host_count': 255,
    'dst_host_srv_count': 10
}

alert = inspect_packet_traffic(suspicious_packet)
print("=== Real-Time Packet Sniffing Result ===")
for k, v in alert.items():
    print(f"{k}: {v}")

# Save Champion Model & Artifacts
joblib.dump(rf_model, 'nids_random_forest_model.pkl')
joblib.dump(scaler, 'nids_scaler.pkl')
print("\n[✓] Model artifacts serialized to disk for production deployment.")